# 03 — Run Qwen 3.5 4B Hybrid RAG & 50-Question Batch Evaluation (Botobot / Group 6)

**Project:** Mapúa University AI3 Machine Project — Philippine Voting Process Helper NLP (`Botobot / Group 6`)  
**Generator Model:** `Qwen/Qwen3.5-4B` (+ optional LoRA adapter `qwen35_4b_lora`)  
**Retriever:** Hybrid `BAAI/bge-m3` (`faiss.IndexFlatIP(1024)`) + `BM25Okapi` fused via **RRF (`k=60`)** + optional Stage-2 `BAAI/bge-reranker-v2-m3` (`Top-20 → Top-5`)  
**Capabilities:** Interactive `ipywidgets` chat UI (Cell 7) + Frozen Top-5 50-question batch evaluation export (Cells 8–10)

In [ ]:
# @title Cell 1: Install Dependencies
!pip -q install -U "transformers>=5.5.0" accelerate peft bitsandbytes faiss-cpu rank-bm25 ipywidgets

In [ ]:
# @title Cell 2: Mount Google Drive, Config Switches & GPU Auto-Guard
import json
import os
from pathlib import Path
import torch

# --- Top-Level Experiment Switches ---
MODEL_ID = "Qwen/Qwen3.5-4B"
MODEL_SLUG = "qwen35"
PRECISION = os.environ.get("AI3_PRECISION", "8bit").lower()             # "8bit" or "16bit"
USE_LORA = os.environ.get("AI3_USE_LORA", "1") == "1"
USE_RERANKER = os.environ.get("AI3_USE_RERANKER", "1") == "1"
ADAPTER_TAG = os.environ.get("AI3_ADAPTER_TAG", "")
USE_DRIVE = os.environ.get("AI3_USE_DRIVE", "1") == "1"

if os.environ.get("AI3_BASE_DIR"):
    BASE_DIR = Path(os.environ["AI3_BASE_DIR"])
elif USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/AI3_Botobot")
    except Exception as exc:
        print(f"[WARN] Drive mount unavailable ({exc}); falling back to /content/AI3_Botobot")
        BASE_DIR = Path("/content/AI3_Botobot")
else:
    BASE_DIR = Path("/content/AI3_Botobot")

INDEX_DIR = BASE_DIR / "data" / "index"
EVAL_DIR = BASE_DIR / "data" / "eval"
ADAPTER_DIR = BASE_DIR / "adapters" / f"qwen35_4b_lora{ADAPTER_TAG}"
OUTPUT_DIR = BASE_DIR / "eval_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- GPU Auto-Guard ---
assert torch.cuda.is_available(), "CUDA GPU is required. Select a T4 or L4 runtime."
GPU_SKU = torch.cuda.get_device_name(0)
MAJOR, MINOR = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

if MAJOR >= 8:
    COMPUTE_DTYPE = torch.bfloat16
else:
    COMPUTE_DTYPE = torch.float16
ATTN_IMPL = None if "gemma-4" in MODEL_ID.lower() else "sdpa"

if PRECISION == "16bit" and VRAM_GB < 14:
    raise RuntimeError(
        f"{MODEL_ID} in 16-bit requires >= 14 GB VRAM (detected {VRAM_GB:.1f} GB on {GPU_SKU}). "
        "Switch to PRECISION='8bit' on T4 or use an L4 (24 GB) runtime."
    )

UNLOAD_RETRIEVER_BEFORE_GEN = (VRAM_GB < 20) or (PRECISION == "16bit" and VRAM_GB < 28)

print(json.dumps({
    "model_id": MODEL_ID,
    "precision": PRECISION,
    "use_lora": USE_LORA,
    "use_reranker": USE_RERANKER,
    "unload_retriever_before_gen": UNLOAD_RETRIEVER_BEFORE_GEN,
    "gpu_sku": GPU_SKU,
    "vram_gb": round(VRAM_GB, 2),
    "compute_dtype": str(COMPUTE_DTYPE),
    "base_dir": str(BASE_DIR),
}, indent=2))

In [ ]:
# @title Cell 3: Load Pre-Built Hybrid Index (FAISS Dense + BM25 Lexical)
import re
import numpy as np
import faiss
from rank_bm25 import BM25Okapi

LEGAL_TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")

def tokenize_legal(text: str):
    return LEGAL_TOKEN_RE.findall(text.lower())

emb_path = INDEX_DIR / "bge_m3_embeddings.npy"
bm25_path = INDEX_DIR / "bm25_tokens.json"
chunks_path = INDEX_DIR / "chunks_index.jsonl"

assert emb_path.exists() and bm25_path.exists() and chunks_path.exists(), (
    f"Missing hybrid index files in {INDEX_DIR}. Upload data/index/ to Drive first."
)

embeddings = np.load(emb_path).astype("float32")
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

bm25_corpus = json.loads(bm25_path.read_text(encoding="utf-8"))
bm25 = BM25Okapi(bm25_corpus)

chunks = [json.loads(line) for line in chunks_path.read_text(encoding="utf-8").splitlines() if line.strip()]

assert index.ntotal == len(chunks) == len(bm25_corpus) == 551, (
    f"Index mismatch: faiss={index.ntotal}, chunks={len(chunks)}, bm25={len(bm25_corpus)}"
)
print(f"Hybrid Index ready: {index.ntotal} chunks | dim={embeddings.shape[1]}")

In [ ]:
# @title Cell 4: Load Query Embedder (BAAI/bge-m3) & Optional Stage-2 Reranker
import gc
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

EMBED_MODEL_ID = "BAAI/bge-m3"
RERANK_MODEL_ID = "BAAI/bge-reranker-v2-m3"
RETRIEVER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_ID, dtype=COMPUTE_DTYPE).to(RETRIEVER_DEVICE)
embed_model.eval()

rerank_tokenizer = None
rerank_model = None
if USE_RERANKER:
    rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_ID)
    rerank_model = AutoModelForSequenceClassification.from_pretrained(
        RERANK_MODEL_ID, dtype=COMPUTE_DTYPE
    ).to(RETRIEVER_DEVICE)
    rerank_model.eval()

def encode_query(query: str) -> np.ndarray:
    enc = embed_tokenizer(
        [query], padding=True, truncation=True, max_length=512, return_tensors="pt"
    ).to(embed_model.device)
    with torch.no_grad():
        out = embed_model(**enc)
        cls_emb = F.normalize(out.last_hidden_state[:, 0], p=2, dim=-1)
    return cls_emb.float().cpu().numpy()

def unload_retriever():
    global embed_model, rerank_model
    if embed_model is not None:
        del embed_model
        embed_model = None
    if rerank_model is not None:
        del rerank_model
        rerank_model = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Retriever models unloaded from GPU.")

print(f"Embedder loaded on {RETRIEVER_DEVICE} | Reranker enabled: {USE_RERANKER}")

In [ ]:
# @title Cell 5: Load Generator Model (16-bit vs 8-bit + Optional LoRA)
import logging
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

if UNLOAD_RETRIEVER_BEFORE_GEN and not (EVAL_DIR / "test_50_questions.jsonl").exists():
    print("[INFO] Keeping retriever loaded for interactive chat; batch eval Cell 9 will offload if needed.")

gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token

if PRECISION == "8bit":
    quant_config = BitsAndBytesConfig(load_in_8bit=True)
else:
    quant_config = None

load_kwargs = dict(
    quantization_config=quant_config,
    dtype=COMPUTE_DTYPE,
    device_map="auto",
)
if ATTN_IMPL is not None:
    load_kwargs["attn_implementation"] = ATTN_IMPL

try:
    gen_model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kwargs)
except Exception:
    gen_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)

# Freeze/skip unused multimodal towers during text-only RAG inference
for attr in ["visual", "vision_tower"]:
    mod = getattr(gen_model, attr, None)
    if mod is not None:
        for p in mod.parameters():
            p.requires_grad = False

if USE_LORA:
    assert ADAPTER_DIR.exists(), f"LoRA adapter not found at {ADAPTER_DIR}. Run trainer notebook first or set USE_LORA=False."
    gen_model = PeftModel.from_pretrained(gen_model, str(ADAPTER_DIR))
    if PRECISION == "16bit":
        gen_model = gen_model.merge_and_unload()
        print("Merged LoRA adapter into 16-bit base weights for zero-overhead latency.")
    else:
        print("Attached LoRA adapter onto 8-bit quantized base weights.")

gen_model.eval()
gen_model.config.use_cache = True
if hasattr(gen_model.config, "text_config") and gen_model.config.text_config is not None:
    gen_model.config.text_config.use_cache = True

print(f"Generator ready: {MODEL_ID} ({PRECISION}, USE_LORA={USE_LORA}) | VRAM={torch.cuda.memory_allocated() / (1024**2):.1f} MiB")

In [ ]:
# @title Cell 6: Hybrid retrieve(), Standardized System Prompt, build_prompt() & answer()
import time

ABSTENTION_TEXT = (
    "I cannot answer this based on the provided 1987 Philippine Constitution "
    "and COMELEC Resolutions No. 11076 and No. 11086."
)

SYSTEM_PROMPT = """You are a precise Philippine election law and COMELEC operations assistant.
Answer strictly from the provided Top-5 context excerpts (from the 1987 Philippine Constitution and COMELEC Resolutions No. 11076 and No. 11086 governing the 12 May 2025 National, Local, and BARMM Parliamentary Elections).
Rules:
1. Always cite the exact source document, Article/Section/Annex/Form number, and inline chunk marker [1]–[5].
2. Preserve all exact numbers, peso amounts, percentages, deadlines, and form codes verbatim.
3. Match the user's language (English or Filipino/Taglish).
4. If the provided context does not contain the answer or falls outside the 12 May 2025 documented scope, respond clearly: "I cannot answer this based on the provided 1987 Philippine Constitution and COMELEC Resolutions No. 11076 and No. 11086." Do not fabricate rules, dates, or citations."""

EOS_IDS = [248044, 248046]

def retrieve(query: str, k: int = 5, candidate_k: int = 20, use_reranker: bool = USE_RERANKER, rrf_k: int = 60):
    assert embed_model is not None, "Embedder is unloaded. Reload Cell 4 or pass frozen_hits."
    q_emb = encode_query(query)
    dense_scores, dense_idxs = index.search(q_emb, candidate_k)
    dense_map = {int(idx): float(sc) for idx, sc in zip(dense_idxs[0], dense_scores[0]) if idx >= 0}

    bm25_scores = bm25.get_scores(tokenize_legal(query))
    bm25_top = np.argsort(bm25_scores)[::-1][:candidate_k]

    rrf = {}
    for rank, idx in enumerate(dense_idxs[0], 1):
        if idx >= 0:
            rrf[int(idx)] = rrf.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(bm25_top, 1):
        if bm25_scores[int(idx)] > 0:
            rrf[int(idx)] = rrf.get(int(idx), 0.0) + 1.0 / (rrf_k + rank)

    candidates = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:candidate_k]
    hits = []
    for idx, rrf_score in candidates:
        rec = dict(chunks[idx])
        rec["dense_score"] = dense_map.get(idx, 0.0)
        rec["bm25_score"] = float(bm25_scores[idx])
        rec["rrf_score"] = float(rrf_score)
        rec["rerank_score"] = None
        hits.append(rec)

    if use_reranker and rerank_model is not None and hits:
        pairs = [[query, h["text"]] for h in hits]
        enc = rerank_tokenizer(
            pairs, padding=True, truncation=True, max_length=1024, return_tensors="pt"
        ).to(rerank_model.device)
        with torch.no_grad():
            logits = rerank_model(**enc).logits.squeeze(-1).float().cpu().tolist()
        if isinstance(logits, float):
            logits = [logits]
        for h, sc in zip(hits, logits):
            h["rerank_score"] = float(sc)
        hits.sort(key=lambda x: x["rerank_score"], reverse=True)

    return hits[:k]

def build_prompt(hits, query: str) -> str:
    context_blocks = []
    for i, h in enumerate(hits, 1):
        p_start, p_end = h.get("page_range", [0, 0])
        pages = f"p. {p_start}" if p_start == p_end else f"pp. {p_start}-{p_end}"
        header = (
            f"[{i}] (chunk_id={h['chunk_id']} | Source: {h['source_file']} | "
            f"{h.get('article', '')} | {h.get('section', '')} | {pages} | "
            f"Scope: {h.get('effective_scope', '')})"
        )
        context_blocks.append(f"{header}\n{h['text']}")
    context = "\n\n".join(context_blocks)
    user_msg = f"Context:\n{context}\n\nQuestion:\n{query}"
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def is_abstention(text: str) -> bool:
    low = text.lower()
    markers = [
        "cannot answer this based on the provided",
        "not covered by the provided",
        "outside the documented scope",
        "hindi saklaw ng",
        "wala sa ibinigay na",
    ]
    return any(m in low for m in markers)

def answer(
    query: str,
    k: int = 5,
    min_dense_score: float = 0.42,
    min_rerank_logit: float = -2.0,
    use_reranker: bool = USE_RERANKER,
    frozen_hits=None,
    max_new_tokens: int = 512,
):
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()

    hits = frozen_hits if frozen_hits is not None else retrieve(query, k=k, use_reranker=use_reranker)
    max_dense = max((h.get("dense_score", 0.0) for h in hits), default=0.0)
    rerank_vals = [h["rerank_score"] for h in hits if h.get("rerank_score") is not None]
    max_rerank = max(rerank_vals) if rerank_vals else 0.0

    # Hard abstention gate when both dense similarity and reranker confidence are weak
    if not hits or (max_dense < min_dense_score and (not rerank_vals or max_rerank < min_rerank_logit)):
        latency_s = time.perf_counter() - t0
        telemetry = {
            "latency_s": round(latency_s, 3),
            "peak_vram_mib": round(torch.cuda.max_memory_allocated() / (1024 ** 2), 1),
            "prompt_tokens": 0,
            "gen_tokens": 0,
            "abstained": True,
            "abstention_source": "retrieval_gate",
        }
        return ABSTENTION_TEXT, hits, telemetry

    prompt = build_prompt(hits, query)
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    prompt_tokens = int(inputs["input_ids"].shape[1])

    with torch.no_grad():
        out_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=EOS_IDS,
            pad_token_id=gen_tokenizer.pad_token_id,
        )

    gen_ids = out_ids[0][prompt_tokens:]
    reply = gen_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    latency_s = time.perf_counter() - t0
    telemetry = {
        "latency_s": round(latency_s, 3),
        "peak_vram_mib": round(torch.cuda.max_memory_allocated() / (1024 ** 2), 1),
        "prompt_tokens": prompt_tokens,
        "gen_tokens": int(gen_ids.shape[0]),
        "abstained": is_abstention(reply),
        "abstention_source": "model" if is_abstention(reply) else "none",
    }
    return reply, hits, telemetry

print("RAG pipeline functions ready.")

In [ ]:
# @title Cell 7: Interactive ipywidgets Chat UI (Upgraded from Sample Notebook)
import ipywidgets as widgets
from IPython.display import display

out = widgets.Output()
inp = widgets.Text(
    placeholder="Ask about Philippine voting rules (English or Taglish)...",
    description="You:",
    layout=widgets.Layout(width="75%"),
)
btn = widgets.Button(description="Send", button_style="primary")
min_dense_slider = widgets.FloatSlider(
    value=0.42, min=0.20, max=0.70, step=0.01,
    description="min_dense:", readout_format=".2f",
    layout=widgets.Layout(width="45%"),
)
rerank_chk = widgets.Checkbox(value=USE_RERANKER, description="Use bge-reranker-v2-m3")

def on_submit(_):
    q = inp.value.strip()
    if not q:
        return
    inp.value = ""
    reply, hits, tel = answer(
        q, k=5, min_dense_score=float(min_dense_slider.value), use_reranker=bool(rerank_chk.value)
    )
    with out:
        print(f"You: {q}\n")
        print(f"Assistant: {reply}\n")
        print(
            f"[Telemetry] Latency: {tel['latency_s']:.2f}s | Peak VRAM: {tel['peak_vram_mib']:.1f} MiB | "
            f"Tokens: {tel['prompt_tokens']} in / {tel['gen_tokens']} out | Abstained: {tel['abstained']}"
        )
        print("Top-5 Retrieved Sources:")
        for i, h in enumerate(hits, 1):
            p0, p1 = h.get("page_range", [0, 0])
            rr = f" rerank={h['rerank_score']:.2f}" if h.get("rerank_score") is not None else ""
            print(f"  [{i}] {h['chunk_id']} | {h['source_file']} | {h.get('section', '')} | pp.{p0}-{p1} | dense={h['dense_score']:.3f} rrf={h['rrf_score']:.4f}{rr}")
        print("\n" + "-" * 90 + "\n")

inp.on_submit(on_submit)
btn.on_click(on_submit)

display(widgets.HBox([inp, btn]))
display(widgets.HBox([min_dense_slider, rerank_chk]))
display(out)

In [ ]:
# @title Cell 8: Generate or Load Frozen Top-5 Retrieval Cache (50 Unseen Questions)
test_q_path = EVAL_DIR / "test_50_questions.jsonl"
cache_path = EVAL_DIR / "frozen_top5_cache.jsonl"

if not test_q_path.exists():
    print(f"[INFO] {test_q_path} not found yet. Author the 50 unseen test questions before running batch eval.")
    frozen_records = []
elif cache_path.exists():
    frozen_records = [json.loads(l) for l in cache_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    print(f"Loaded existing frozen Top-5 cache: {len(frozen_records)} questions from {cache_path}")
else:
    test_questions = [json.loads(l) for l in test_q_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    frozen_records = []
    for item in test_questions:
        hits = retrieve(item["question"], k=5, use_reranker=USE_RERANKER)
        frozen_records.append({**item, "frozen_top5_hits": hits})
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with cache_path.open("w", encoding="utf-8") as f:
        for rec in frozen_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Generated and saved frozen Top-5 cache ({len(frozen_records)} queries) to {cache_path}")

In [ ]:
# @title Cell 9: 50-Question Batch Evaluation Runner
out_jsonl = OUTPUT_DIR / f"{MODEL_SLUG}_{PRECISION}_preds.jsonl"

if not frozen_records:
    print("[SKIP] No frozen_records loaded in Cell 8.")
else:
    if UNLOAD_RETRIEVER_BEFORE_GEN:
        unload_retriever()

    results = []
    for idx, rec in enumerate(frozen_records, 1):
        hits = rec["frozen_top5_hits"]
        reply, _, tel = answer(rec["question"], k=5, frozen_hits=hits)
        row = {
            "id": rec.get("id", f"q_{idx:02d}"),
            "question": rec["question"],
            "is_answerable": bool(rec.get("is_answerable", True)),
            "category": rec.get("category", ""),
            "ground_truth": rec.get("ground_truth", ""),
            "gold_chunk_ids": rec.get("gold_chunk_ids", []),
            "retrieved_chunk_ids": [h["chunk_id"] for h in hits],
            "retrieved_contexts": [h["text"] for h in hits],
            "answer": reply,
            "abstained": tel["abstained"],
            "abstention_source": tel["abstention_source"],
            "latency_s": tel["latency_s"],
            "peak_vram_mib": tel["peak_vram_mib"],
            "prompt_tokens": tel["prompt_tokens"],
            "gen_tokens": tel["gen_tokens"],
            "gpu_sku": GPU_SKU,
            "precision": PRECISION,
            "use_lora": USE_LORA,
            "model_id": MODEL_ID,
        }
        results.append(row)
        if idx % 10 == 0 or idx == len(frozen_records):
            print(f"Processed {idx}/{len(frozen_records)} | last_latency={tel['latency_s']:.2f}s | abstained={tel['abstained']}")

    with out_jsonl.open("w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(results)} predictions to {out_jsonl}")

In [ ]:
# @title Cell 10: Quick Run Summary & Export Check
import re

if not out_jsonl.exists():
    print("[INFO] Run Cell 9 first once data/eval/test_50_questions.jsonl is uploaded.")
else:
    preds = [json.loads(l) for l in out_jsonl.read_text(encoding="utf-8").splitlines() if l.strip()]
    ans_rows = [p for p in preds if p["is_answerable"]]
    unans_rows = [p for p in preds if not p["is_answerable"]]

    correct_abst = sum(1 for p in unans_rows if p["abstained"]) / max(len(unans_rows), 1)
    false_abst = sum(1 for p in ans_rows if p["abstained"]) / max(len(ans_rows), 1)
    latencies = [p["latency_s"] for p in preds]
    max_vram = max((p["peak_vram_mib"] for p in preds), default=0.0)
    cite_re = re.compile(r"(\[[1-5]\]|SEC\.|SECTION|Article|Annex|CEF)", re.IGNORECASE)
    cite_cov = sum(1 for p in ans_rows if not p["abstained"] and cite_re.search(p["answer"])) / max(len(ans_rows), 1)

    summary = {
        "output_file": str(out_jsonl),
        "model_id": MODEL_ID,
        "precision": PRECISION,
        "use_lora": USE_LORA,
        "gpu_sku": GPU_SKU,
        "total_questions": len(preds),
        "answerable_count": len(ans_rows),
        "unanswerable_count": len(unans_rows),
        "correct_abstention_rate": round(correct_abst, 4),
        "false_abstention_rate": round(false_abst, 4),
        "avg_response_latency_s": round(float(np.mean(latencies)), 3),
        "p95_response_latency_s": round(float(np.percentile(latencies, 95)), 3),
        "max_peak_vram_mib": round(max_vram, 1),
        "citation_coverage_rate": round(cite_cov, 4),
    }
    print(json.dumps(summary, indent=2))